<a href="https://colab.research.google.com/github/soldatovde1979-ai/ReportSPPR/blob/main/RepSSPR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# БЛОК 0 и 1: ГЛОБАЛЬНЫЕ ПАРАМЕТРЫ, АВТОРИЗАЦИЯ, ФИЛЬТР

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 1] БЛОК 0 и 1: ГЛОБАЛЬНЫЕ ПАРАМЕТРЫ, АВТОРИЗАЦИЯ, ФИЛЬТР И ЗАГРУЗКА
# =====================================================================
import os
import io
import glob
import time
import json
import math
import asyncio
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field
from jinja2 import Template
from google import genai
from google.colab import drive, files, userdata
from openai import AsyncOpenAI, OpenAI

# --- ПАРАМЕТРЫ ПРОВАЙДЕРА И МОДЕЛЕЙ ---
AI_PROVIDER = "deepseek" #@param ["google", "deepseek"]

# Google Gemini модели
GEMINI_BATCH_MODEL = 'gemini-3.6-flash'
GEMINI_SUMMARY_MODEL = 'gemini-3.1-pro'

# DeepSeek модели
DEEPSEEK_BATCH_MODEL = 'deepseek-chat'
DEEPSEEK_SUMMARY_MODEL = 'deepseek-reasoner'

# Автоматический выбор активных моделей
BATCH_MODEL = GEMINI_BATCH_MODEL if AI_PROVIDER == "google" else DEEPSEEK_BATCH_MODEL
SUMMARY_MODEL = GEMINI_SUMMARY_MODEL if AI_PROVIDER == "google" else DEEPSEEK_SUMMARY_MODEL

BATCH_SIZE = 40
MAX_CONCURRENCY = 4
SLEEP_TIME = 2

# --- РЕЖИМ ВЫБОРА ФАЙЛА ---
USE_FILE_PICKER = True #@param {type:"boolean"}

# --- ПАРАМЕТРЫ ТЕСТИРОВАНИЯ ---
TEST_SAMPLE_SIZE = 0

# --- ПАРАМЕТРЫ БИЗНЕС-ЛОГИКИ ---
WORK_HOURS = 8
USER_APPROVAL_DAYS = 3
APPROVAL_HOURS_DEDUCT = USER_APPROVAL_DAYS * WORK_HOURS  # 24 рабочих часа
TOP_INITIATORS_N = 5
TOP_LONGEST_N = 10
TOP_PROBLEMS_N = 5

# --- ПАРАМЕТРЫ ФАЙЛОВОЙ СИСТЕМЫ ---
SOURCE_DIR_VARIANTS = [
    '/content/drive/MyDrive/REPORTS/SPPR/source',
    '/content/drive/MyDrive/reports/sppr/source'
]
TARGET_DIR = '/content/drive/MyDrive/REPORTS/SPPR/result'
HISTORY_DIR = os.path.join(TARGET_DIR, 'history')

# --- ИНИЦИАЛИЗАЦИЯ ИИ КЛИЕНТОВ И ДИСКА ---
# 1. Google Gemini
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client_gemini = genai.Client(api_key=GOOGLE_API_KEY.strip()) if GOOGLE_API_KEY else None

# 2. DeepSeek (через OpenAI SDK)
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY') or "sk-a4944807fdf94c8b91813c2ece736419"

client_deepseek_async = AsyncOpenAI(
    api_key=DEEPSEEK_API_KEY.strip(),
    base_url="https://api.deepseek.com/v1"
)
client_deepseek_sync = OpenAI(
    api_key=DEEPSEEK_API_KEY.strip(),
    base_url="https://api.deepseek.com/v1"
)

if AI_PROVIDER == "google" and not GOOGLE_API_KEY:
    raise ValueError("Выбран Google, но ключ GEMINI_API_KEY отсутствует.")

# Совместимость: универсальный клиент под выбранный провайдер
client = client_gemini if AI_PROVIDER == "google" else client_deepseek_sync

# Монтирование Google Диска и создание каталогов
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=False)

os.makedirs(TARGET_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

print(f"🤖 Активный провайдер: {AI_PROVIDER.upper()} | Батч-модель: {BATCH_MODEL} | Аналитическая: {SUMMARY_MODEL}")

# --- ЗАГРУЗКА ДАТАСЕТА (РАБОТА С ФАЙЛАМИ) ---
if USE_FILE_PICKER:
    print("📁 Выберите файл выгрузки (CSV или JSON) для загрузки:")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Файл не был выбран.")

    file_name = list(uploaded.keys())[0]
    file_bytes = list(uploaded.values())[0]
    print(f"📥 Загружен локальный файл: {file_name}")

    if file_name.endswith('.json'):
        df = pd.read_json(io.BytesIO(file_bytes))
    else:
        df = pd.read_csv(io.BytesIO(file_bytes), sep=';', encoding='utf-8')
else:
    SOURCE_DIR = next(
        (path for path in SOURCE_DIR_VARIANTS if os.path.exists(path) and glob.glob(os.path.join(path, '*.*'))),
        None
    )
    if not SOURCE_DIR:
        raise FileNotFoundError("Исходные директории на Google Диске не найдены.")

    latest_file = max(
        glob.glob(os.path.join(SOURCE_DIR, '*.csv')) + glob.glob(os.path.join(SOURCE_DIR, '*.json')),
        key=os.path.getmtime
    )
    print(f"📄 Загрузка последнего файла с Google Диска: {latest_file}")
    df = pd.read_json(latest_file) if latest_file.endswith('.json') else pd.read_csv(latest_file, sep=';', encoding='utf-8')

# --- ПАРСИНГ ДАТ И ФИЛЬТРАЦИЯ ---
df['reg_dt'] = pd.to_datetime(df['reg'], errors='coerce')
df['cls_dt'] = pd.to_datetime(df['cls'], errors='coerce')

df_closed = df.dropna(subset=['cls_dt']).copy()
df_closed['iso_year'] = df_closed['cls_dt'].dt.isocalendar().year
df_closed['iso_week'] = df_closed['cls_dt'].dt.isocalendar().week

max_year = df_closed['iso_year'].max()
max_week = df_closed[df_closed['iso_year'] == max_year]['iso_week'].max()

df = df_closed.copy()

if TEST_SAMPLE_SIZE > 0:
    df = df.head(TEST_SAMPLE_SIZE).copy()
    print(f"🧪 ТЕСТ-РЕЖИМ: Взято {len(df)} строк. История не пишется.")
else:
    print(f"✂️ РАБОЧИЙ РЕЖИМ (По дате закрытия): Неделя {max_year}-W{max_week}. Итого строк в обработке: {len(df)}")

🤖 Активный провайдер: DEEPSEEK | Батч-модель: deepseek-chat | Аналитическая: deepseek-reasoner
📁 Выберите файл выгрузки (CSV или JSON) для загрузки:


Saving sppr_dump_20260826_151638_139rec (1).json to sppr_dump_20260826_151638_139rec (1) (1).json
📥 Загружен локальный файл: sppr_dump_20260826_151638_139rec (1) (1).json
✂️ РАБОЧИЙ РЕЖИМ (По дате закрытия): Неделя 2026-W34. Итого строк в обработке: 139


# БЛОК 2: ВЕКТОРНЫЙ РАСЧЕТ SLA

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 2] БЛОК 2: ВЕКТОРНЫЙ РАСЧЕТ SLA
# =====================================================================
# 🎛️ ГЛАВНЫЙ ПЕРЕКЛЮЧАТЕЛЬ 🎛️
ENABLE_MAIN = True #@param {type:"boolean"}

if ENABLE_MAIN:
    df['safe_cls'] = df['cls_dt'].fillna(pd.Timestamp.now())
    dates_reg = df['reg_dt'].dt.normalize().values.astype('datetime64[D]')
    dates_cls = df['safe_cls'].dt.normalize().values.astype('datetime64[D]')

    # Расчет рабочих часов за вычетом ожидания согласования (3 дня * 8 часов = 24 часа)
    raw_work_hours = np.busday_count(dates_reg, dates_cls) * WORK_HOURS
    df['dif_Hour'] = np.maximum(0, raw_work_hours - APPROVAL_HOURS_DEDUCT)

    print(f"✅ SLA рассчитан (с вычетом {APPROVAL_HOURS_DEDUCT} ч. ожидания). Медиана: {df['dif_Hour'].median():.1f} ч.")
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

✅ SLA рассчитан (с вычетом 24 ч. ожидания). Медиана: 0.0 ч.


# БЛОК 3 и 4: АСИНХРОННАЯ ПАКЕТНАЯ ОБРАБОТКА

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 3] БЛОК 3 и 4: АСИНХРОННАЯ ПАКЕТНАЯ ОБРАБОТКА
# =====================================================================
class TicketResult(BaseModel):
    id: str = Field(description="Идентификатор тикета")
    topic_cluster: str = Field(description="Конкретная суть сбоя/операции. Без абстракций.")
    tone_deviation_score: int = Field(description="Шкала тона от 0 до 3")
    is_emotional: bool = Field(description="True если tone_deviation_score > 0")
    emotion_argument: str = Field(description="Обоснование оценки тона")
    complexity: int = Field(description="Сложность задачи от 1 до 5")
    initial_grade: str = Field(description="Исходная оценка")
    final_grade: str = Field(description="Итоговый грейд: ПЛОХО, ХОРОШО, ОТЛИЧНО")
    grade_argument: str = Field(description="Детальная аргументация грейда и причин снижения")
    is_alternative: bool = Field(description="True если ИИ не согласен с закрытием тикета")
    is_critical_incident: bool = Field(description="True если инцидент блокирует бизнес-процесс")
    planshet: bool = Field(description="True Если Обращение относится к работе планшетов")
class BatchResponse(BaseModel):
    results: list[TicketResult]

if ENABLE_MAIN:
    SYSTEM_INSTRUCTION = """
    Вы — ведущий системный аналитик и архитектор 1С. Ваша задача — провести жесткий, технически объективный аудит пакета обращений в службу поддержки.

    1. ПРАВИЛА КЛАСТЕРИЗАЦИИ (topic_cluster):
    - Категорически запрещены абстрактные формулировки: "Ошибка системы", "Проблема с 1С", "Консультация", "Вопрос по ДО".
    - Указывайте КОНКРЕТНУЮ суть сбоя или операции. Пример: "Ошибка при проведении документа Заказ", "Неверный расчет лимита БТД", "Зависание сессии при экспорте".

    2. КРИТЕРИИ ОЦЕНКИ ГРЕЙДА (final_grade):
    - ПЛОХО:
      * Решение содержит прямые ручные правки в базе данных (SQL/1С) без устранения причины.
      * Аналитик перенаправил тикет без ответа или закрыл с формулировкой "дубликат/ошибка" без объяснений.
      * Ответ не решает проблему пользователя, формальная отписка.
      * Время решения несоразмерно задаче (простая консультация висела несколько дней).
      * Удаление ЭЦП если нет согласования пользователя
    - ХОРОШО:
      * Стандартное корректное решение, проблема решена, даны исчерпывающие пояснения.
      * Удаление ЭЦП если есть согласование пользователя
      * Обращения с темой "Закрытие отчетного периода", "Встречи", "Планерка" являются ХОРОШО
      * Обращения содержащие примерный текст "Ваше обращение передано в Сервис Деск. Запрос на обслуживание номер" - это норма, значит не наше обращение
    - ОТЛИЧНО:
      * Превентивное решение. Аналитик не просто исправил ошибку, но и завел задачу на баг-трекер, предложил автоматизацию или дал инструкцию, исключающую повторение.

    3. ШКАЛА ЭМОЦИОНАЛЬНОГО ТОНА (tone_deviation_score):
    - 0 (Деловой): Конструктивное описание проблемы, стандартный рабочий диалог.
    - 1 (Скрытое недовольство): Ирония, сарказм, жалобы на длительность ("Сколько можно ждать", "Опять ничего не работает").
    - 2 (Явная агрессия): Требования эскалации, капслок в претензиях, угрозы жалоб руководству.
    - 3 (Оскорбления): Прямая ругань, переходы на личности.

    ИСКЛЮЧЕНИЯ (ОБЯЗАТЕЛЬНО СТАВИТЬ ТОН 0):
    - Множественные знаки препинания в приветствиях или эмодзи (например: "Добрый день!!!", "Спасибо!").
    - Написание аббревиатур и названий систем CAPS LOCK (например: "Ошибки в ЗУП", "Не проводится ГПД").
    - Использование слова "СРОЧНО" или "КРИТИЧНО", если бизнес-процесс действительно заблокирован.

    4. СЛОЖНОСТЬ (complexity):
    - Оценка от 1 (простая консультация/кнопка) до 5 (комплексный сбой архитектуры/перерасчет регистров).

    5. АЛЬТЕРНАТИВНОЕ МНЕНИЕ (is_alternative):
    - Ставьте true, если первоначальный статус тикета или ответ аналитика формально закрывает обращение, но по существу проблема пользователя НЕ решена, или решение содержит критическую ошибку/риск для системы.

    6. КРИТИЧЕСКИЙ ИНЦИДЕНТ (is_critical_incident):
    - Ставьте true, если сбой заблокировал работу подразделения, вызвал остановку отгрузок/расчета зарплаты или затронул более 5 пользователей одновременно.

    Перед вынесением итоговых оценок ОБЯЗАТЕЛЬНО заполняйте аргументацию (emotion_argument, grade_argument), подробно объясняя причину снижения грейда или фиксации эмоционального отклонения.
    """

    async def call_ai_provider_batch(prompt: str):
        """Универсальный вызов для Google и DeepSeek"""
        if AI_PROVIDER == "google":
            response = await client_gemini.aio.models.generate_content(
                model=BATCH_MODEL, contents=prompt,
                config={
                    'system_instruction': SYSTEM_INSTRUCTION,
                    'response_mime_type': 'application/json',
                    'response_schema': BatchResponse,
                    'temperature': 0.1
                }
            )
            tokens = {
                'prompt': response.usage_metadata.prompt_token_count if response.usage_metadata else 0,
                'candidates': response.usage_metadata.candidates_token_count if response.usage_metadata else 0,
                'total': response.usage_metadata.total_token_count if response.usage_metadata else 0
            }
            data = json.loads(response.text).get('results', [])
            return data, tokens

        else: # deepseek
            schema_format = BatchResponse.model_json_schema()
            full_system = f"{SYSTEM_INSTRUCTION}\nОтвет строго в формате JSON по схеме:\n{json.dumps(schema_format, ensure_ascii=False)}"

            response = await client_deepseek_async.chat.completions.create(
                model=BATCH_MODEL,
                messages=[
                    {"role": "system", "content": full_system},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.1
            )
            usage = response.usage
            tokens = {
                'prompt': usage.prompt_tokens if usage else 0,
                'candidates': usage.completion_tokens if usage else 0,
                'total': usage.total_tokens if usage else 0
            }
            res_raw = json.loads(response.choices[0].message.content)
            data = res_raw.get('results', []) if isinstance(res_raw, dict) else res_raw
            return data, tokens

    async def process_chunk(chunk_id, chunk, sem):
        prompt_rows = []
        for _, r in chunk.iterrows():
            prompt_rows.append(
                f"ID: {r['id']} | Инициатор: {r.get('cli','')} | Аналитик: {r.get('anl','')}\n"
                f"Тема: {r.get('sec','')}\n"
                f"Описание: {r.get('desc','')}\n"
                f"Решение: {r.get('sol','')}\n"
                f"----------------------------------------"
            )
        prompt = "Оцени следующий пакет обращений:\n" + "\n".join(prompt_rows)

        async with sem:
            max_retries = 4
            delay = SLEEP_TIME
            for attempt in range(1, max_retries + 1):
                try:
                    await asyncio.sleep(delay)
                    start_t = time.time()
                    data, tokens = await call_ai_provider_batch(prompt)
                    print(f"✅ Пакет {chunk_id} завершен ({time.time() - start_t:.1f}s, Токенов: {tokens['total']})")
                    return data, tokens
                except Exception as e:
                    print(f"⚠️ Сбой пакета {chunk_id} (Попытка {attempt}/{max_retries}): {e}")
                    if attempt == max_retries:
                        fallback_results = [
                            {
                                'id': r['id'],
                                'topic_cluster': str(r.get('sec', 'Не определено')).strip() or 'Без темы',
                                'tone_deviation_score': 0,
                                'is_emotional': False,
                                'emotion_argument': '',
                                'complexity': 1,
                                'initial_grade': 'ХОРОШО',
                                'final_grade': 'ХОРОШО',
                                'grade_argument': f'Ошибка вызова API ({AI_PROVIDER}): {e}',
                                'is_alternative': False,
                                'is_critical_incident': False
                            } for _, r in chunk.iterrows()
                        ]
                        return fallback_results, {'prompt': 0, 'candidates': 0, 'total': 0}
                    delay *= 2

    async def run_batching():
        sem = asyncio.Semaphore(MAX_CONCURRENCY)
        tasks = []
        chunks = [df.iloc[i:i+BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]

        print(f"🚀 Старт асинхронного анализа (Модель: {BATCH_MODEL} | Пакетов: {len(chunks)} | Потоков: {MAX_CONCURRENCY})...")
        start_total = time.time()

        for idx, chunk in enumerate(chunks, 1):
            tasks.append(process_chunk(idx, chunk, sem))

        results = await asyncio.gather(*tasks)

        flat_results = []
        total_batch_tokens = {'prompt': 0, 'candidates': 0, 'total': 0}

        for res_list, toks in results:
            flat_results.extend(res_list)
            total_batch_tokens['prompt'] += toks['prompt']
            total_batch_tokens['candidates'] += toks['candidates']
            total_batch_tokens['total'] += toks['total']

        print(f"✅ Анализ завершен за {time.time() - start_total:.1f} сек. Итого токенов батчинга: {total_batch_tokens['total']}")
        return flat_results, total_batch_tokens

    # Запуск
    ai_results, total_batch_tokens = await run_batching()
    df_final = pd.merge(df, pd.DataFrame(ai_results), on='id', how='left')
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

🚀 Старт асинхронного анализа (Модель: deepseek-chat | Пакетов: 4 | Потоков: 4)...


CancelledError: 

# БЛОК 5: АГРЕГАЦИЯ И ИСТОРИЯ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 4] БЛОК 5: АГРЕГАЦИЯ И ИСТОРИЯ
# =====================================================================
if ENABLE_MAIN:
    df_final['final_grade'] = df_final.get('final_grade', pd.Series(['ХОРОШО']*len(df_final))).fillna('ХОРОШО').astype(str).str.upper().str.strip()
    for col in ['complexity', 'tone_deviation_score']: df_final[col] = pd.to_numeric(df_final.get(col, 1), errors='coerce').fillna(1 if col=='complexity' else 0)
    for col in ['is_critical_incident', 'is_alternative', 'planshet']: df_final[col] = df_final.get(col, False).astype(bool)


    # Полная очистка датафрейма для исключения "nan" в таблицах
    df_clean = df_final.fillna('')

    total_tickets = len(df_clean)
    min_cls_str = df_clean['cls_dt'].min().strftime('%d.%m.%Y') if pd.notnull(df_clean['cls_dt'].min()) else 'Н/Д'
    medHour = round(df_clean['dif_Hour'].median(), 1) if not df_clean.empty else 0.0

    # Топы (подготовка данных для Jinja2)
    top_initiators = df_clean.groupby('cli').agg(
        count=('id', 'count'),
        topics=('topic_cluster', lambda x: ', '.join(x.dropna().unique()[:3])),
        tickets=('id', lambda x: ', '.join(x.astype(str).tolist()))
    ).reset_index().sort_values('count', ascending=False).head(TOP_INITIATORS_N).to_dict('records')

    top_topics = df_clean.groupby('topic_cluster').agg(
        count=('id', 'count'),
        tickets=('id', lambda x: ', '.join(x.astype(str).tolist())), # Выводим ВСЕ тикеты без ограничения [:5]
        quality=('final_grade', lambda x: x.mode()[0] if not x.mode().empty else 'Н/Д'),
        sample_desc=('desc', lambda x: str(x.dropna().iloc[0]).strip().replace('\n', ' ') if not x.dropna().empty else 'Нет описания') # Текст первого обращения
    ).reset_index().sort_values('count', ascending=False).head(TOP_PROBLEMS_N).to_dict('records')

    longest_tickets = df_clean.sort_values('dif_Hour', ascending=False).head(TOP_LONGEST_N).to_dict('records')
    emo_tickets = df_clean[df_clean['tone_deviation_score'] > 0].sort_values('tone_deviation_score', ascending=False).to_dict('records')
    crit_tickets = df_clean[df_clean['is_critical_incident']].sort_values('reg_dt').to_dict('records')
    bad_tickets = df_clean[df_clean['final_grade'] == 'ПЛОХО'].head(10).to_dict('records')
    # ----------------------------------------------------------------------
    # ЧТО ИЗМЕНЕНО: Добавлена выборка обращений по планшетам
    # ПОЧЕМУ: Формирует список записей (словарей) для передачи в HTML-шаблон
    # ----------------------------------------------------------------------
    planshet_tickets = df_clean[df_clean['planshet']].sort_values('reg_dt', ascending=False).to_dict('records')
    planshet_total = len(planshet_tickets)

    # Честный подсчет плохих тикетов (без ограничения .head(10))
    bad_count = len(df_clean[df_clean['final_grade'] == 'ПЛОХО'])

    # История: сначала читаем прошлую
    hist_files = sorted(glob.glob(os.path.join(HISTORY_DIR, "week_*.json")))
    hist_data = [json.load(open(hf, 'r', encoding='utf-8')) for hf in hist_files]
    prev_week_data = hist_data[-1] if len(hist_data) > 0 else None
    w1_t = prev_week_data['total_tickets'] if prev_week_data else None
    w1_m = prev_week_data['med_hour'] if prev_week_data else None

    # Записываем текущую
    if TEST_SAMPLE_SIZE == 0:
        current_week_data = {"year": int(max_year), "week_num": int(max_week), "total_tickets": int(total_tickets), "med_hour": float(medHour)}
        with open(os.path.join(HISTORY_DIR, f"week_{max_year}_W{max_week:02d}.json"), 'w', encoding='utf-8') as f:
            json.dump(current_week_data, f, ensure_ascii=False, indent=2)
        hist_data.append(current_week_data) # Добавляем в массив для графиков

    # Контекст для Gemini Pro
    bad_summary_text = "\n".join([f"- ID {r['id']} ({r['topic_cluster']}): {r['desc']} | Ответ: {r['sol']} | Ошибка: {r['grade_argument']}" for r in bad_tickets[:5]])
    crit_summary_text = "\n".join([f"- ID {r['id']}: {r['topic_cluster']} ({r['desc']})" for r in crit_tickets[:5]])
    top_topics_text = ", ".join([f"{t['topic_cluster']} ({t['count']} шт)" for t in top_topics[:5]])

    summary_for_ai = f"""СТАТИСТИКА НЕДЕЛИ:
    - Всего обращений: {total_tickets}
    - Медианное время решения: {medHour} ч.
    - Топ проблемных кластеров: {top_topics_text}
    - Критических инцидентов: {len(crit_tickets)}
    - Обращений с оценкой ПЛОХО: {bad_count}
    КЛЮЧЕВЫЕ КРИТИЧЕСКИЕ СБОИ:
    {crit_summary_text if crit_summary_text else 'Нет'}
    ПРИМЕРЫ ПЛОХИХ РЕШЕНИЙ АНАЛИТИКОВ:
    {bad_summary_text if bad_summary_text else 'Нет'}"""

    summary_tokens = {'prompt': 0, 'candidates': 0, 'total': 0}
    summary_prompt = f"Ты — главный архитектор систем 1С. Дай 3 глубоких вывода об архитектуре процесса и Root Cause: {summary_for_ai}. Формат HTML <li>"

    try:
        print(f"🧠 Генерация выводов Архитектора ({AI_PROVIDER.upper()} -> {SUMMARY_MODEL})...")
        if AI_PROVIDER == "google":
            response = client_gemini.models.generate_content(
                model=SUMMARY_MODEL,
                contents=summary_prompt,
                config={'temperature': 0.2}
            )
            if response.usage_metadata:
                summary_tokens['prompt'] = response.usage_metadata.prompt_token_count
                summary_tokens['candidates'] = response.usage_metadata.candidates_token_count
                summary_tokens['total'] = response.usage_metadata.total_token_count
            ai_conclusions = response.text.replace('```html', '').replace('```', '').strip()
        else: # deepseek
            response = client_deepseek_sync.chat.completions.create(
                model=SUMMARY_MODEL,
                messages=[
                    {"role": "system", "content": "Ты — ведущий аналитик и архитектор 1С. Отвечай только набором HTML-тегов <li> без внешних блоков markdown."},
                    {"role": "user", "content": summary_prompt}
                ],
                temperature=0.2
            )
            usage = response.usage
            if usage:
                summary_tokens['prompt'] = usage.prompt_tokens
                summary_tokens['candidates'] = usage.completion_tokens
                summary_tokens['total'] = usage.total_tokens
            ai_conclusions = response.choices[0].message.content.replace('```html', '').replace('```', '').strip()

        print(f"✅ Выводы сгенерированы. Потреблено токенов ({SUMMARY_MODEL}): {summary_tokens['total']}")
    except Exception as e:
        ai_conclusions = f"<li>Ошибка генерации выводов ({AI_PROVIDER}): {e}</li>"
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

🧠 Генерация выводов Архитектора (DEEPSEEK -> deepseek-reasoner)...
✅ Выводы сгенерированы. Потреблено токенов (deepseek-reasoner): 4824


# БЛОК 5.1. Логирование

In [ ]:
# =====================================================================
# БЛОК 5.1: АВТОМАТИЧЕСКОЕ ЛОГИРОВАНИЕ В GOOGLE SHEETS (LOGS & TOKEN)
# =====================================================================
import datetime
import gspread
from google.colab import auth
from google.auth import default

if ENABLE_MAIN:
    # 1. Авторизация в Google Sheets API
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # 2. Поиск и открытие файла 'Logs' в директории TARGET_DIR
    LOGS_FILE_NAME = 'Logs'
    logs_path = os.path.join(TARGET_DIR, LOGS_FILE_NAME)

    try:
        # Открываем существующую таблицу по имени в целевой папке
        sh = gc.open(LOGS_FILE_NAME)
    except Exception as e:
        raise FileNotFoundError(f"Файл таблицы '{LOGS_FILE_NAME}' не найден в Google Sheets. Проверьте права и имя.")

    # 3. Генерация уникального Run ID
    run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # 4. Запись в лист 'Logs'
    try:
        ws_logs = sh.worksheet('Logs')
    except gspread.exceptions.WorksheetNotFound:
        ws_logs = sh.add_worksheet(title='Logs', rows=100, cols=10)
        ws_logs.append_row(['Run_ID', 'Timestamp', 'Year', 'Week', 'Total_Tickets', 'Status'])

    ws_logs.append_row([
        run_id,
        datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        int(max_year),
        int(max_week),
        len(df_clean),
        "SUCCESS"
    ])

    # 5. Запись в лист 'Token'
    try:
        ws_tokens = sh.worksheet('Token')
    except gspread.exceptions.WorksheetNotFound:
        ws_tokens = sh.add_worksheet(title='Token', rows=100, cols=10)
        ws_tokens.append_row([
            'Run_ID',
            'Batch_Model', 'Batch_Prompt_Tokens', 'Batch_Candidates_Tokens', 'Batch_Total_Tokens',
            'Summary_Model', 'Summary_Prompt_Tokens', 'Summary_Candidates_Tokens', 'Summary_Total_Tokens'
        ])

    # Запись точной статистики токенов из словарей, собранных в Ячейках 3 и 4
    ws_tokens.append_row([
        run_id,
        BATCH_MODEL,
        total_batch_tokens['prompt'],
        total_batch_tokens['candidates'],
        total_batch_tokens['total'],
        SUMMARY_MODEL,
        summary_tokens['prompt'],
        summary_tokens['candidates'],
        summary_tokens['total']
    ])

    print(f"📊 Лог и токены успешно записаны в 'Logs' (Run ID: {run_id})")
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

📊 Лог и токены успешно записаны в 'Logs' (Run ID: 20260826_124818)


# БЛОК 6: JINJA2 РЕНДЕРИНГ И ЭКСПОРТ

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 5] БЛОК 6: JINJA2 РЕНДЕРИНГ И ЭКСПОРТ
# =====================================================================
import os
import json
import pandas as pd
from jinja2 import Template
from google.colab import drive

if ENABLE_MAIN:
    root_search_dir = '/content/drive/MyDrive/REPORTS/SPPR'

    def find_template(search_dir):
        """Ищет template.html в директории."""
        if os.path.exists(search_dir):
            for dirpath, _, filenames in os.walk(search_dir):
                for f in filenames:
                    if f.lower() == 'template.html':
                        return os.path.join(dirpath, f)
        return None

    # 1. Первая попытка поиска
    TEMPLATE_PATH = find_template(root_search_dir)

    # 2. Если кэш Colab устарел и файл не найден — сбрасываем и переподключаем Диск
    if not TEMPLATE_PATH:
        print("⚠️ Файл template.html не найден в кэше. Сбрасываем кэш Google Диска...")
        try:
            drive.flush_and_unmount()
        except Exception:
            pass

        drive.mount('/content/drive', force_remount=True)

        # Повторный поиск после очистки кэша
        TEMPLATE_PATH = find_template(root_search_dir)

    if not TEMPLATE_PATH:
        raise FileNotFoundError(
            f"❌ Файл template.html не найден в {root_search_dir} даже после сброса кэша.\n"
            f"Убедитесь, что файл не переименован (например, в template.html.txt или template (1).html)."
        )

    print(f"✅ Шаблон найден и загружен из: {TEMPLATE_PATH}")

    with open(TEMPLATE_PATH, 'r', encoding='utf-8') as f:
        template = Template(f.read())


    def badge_class(val):
        return 'excellent' if val == 'ОТЛИЧНО' else 'good' if val == 'ХОРОШО' else 'bad' if val == 'ПЛОХО' else 'orange'

    # Досборка недостающих переменных (используем df_clean, защищенный от nan)
    min_reg_str = df_clean['reg_dt'].min().strftime('%d.%m.%Y') if pd.notnull(df_clean['reg_dt'].min()) else 'Н/Д'
    max_reg_str = df_clean['reg_dt'].max().strftime('%d.%m.%Y') if pd.notnull(df_clean['reg_dt'].max()) else 'Н/Д'
    max_cls_str = df_clean['cls_dt'].max().strftime('%d.%m.%Y') if pd.notnull(df_clean['cls_dt'].max()) else 'Н/Д'
    midHour = round(df_clean['dif_Hour'].mean(), 1) if not df_clean.empty else 0.0

    emo_total = len(df_clean[df_clean['tone_deviation_score'] > 0])
    emo_avg = round(df_clean[df_clean['tone_deviation_score'] > 0]['tone_deviation_score'].mean(), 1) if emo_total > 0 else 0
    emo_max = df_clean['tone_deviation_score'].max() if emo_total > 0 else 0

    alt_df = df_clean[df_clean['is_alternative']]
    alt_total = len(alt_df)
    alt_tickets = alt_df.head(10).to_dict('records')
    exc_tickets = df_clean[df_clean['final_grade'] == 'ОТЛИЧНО'].head(5).to_dict('records')
    last30_tickets = df_clean.sort_values('reg_dt', ascending=False).head(30).to_dict('records')
    hardest_tickets = df_clean.sort_values('complexity', ascending=False).head(5).to_dict('records')
    easiest_tickets = df_clean.sort_values('complexity', ascending=True).head(5).to_dict('records')

    grade_counts = pd.crosstab(df_clean['anl'], df_clean['final_grade'])
    for col in ['ОТЛИЧНО', 'ХОРОШО', 'ПЛОХО']:
        if col not in grade_counts: grade_counts[col] = 0
    analysts = grade_counts.join(df_clean.groupby('anl').agg(comp=('complexity', 'mean'), total=('id', 'count')))
    analysts['score'] = (analysts['ОТЛИЧНО'] * 3 + analysts['ХОРОШО'] * 1 - analysts['ПЛОХО'] * 3) * analysts['comp']
    analysts = analysts.reset_index().sort_values('score', ascending=False)
    top_analysts = analysts.head(10).to_dict('records')

    nom_qty = analysts.sort_values('total', ascending=False).iloc[0] if not analysts.empty else None
    nom_qual = analysts.sort_values('score', ascending=False).iloc[0] if not analysts.empty else None
    nom_hard_ticket = df_clean.sort_values('complexity', ascending=False).iloc[0] if not df_clean.empty else None

    # Подготовка JSON для графиков
    pie_json = json.dumps(df_clean['topic_cluster'].value_counts().to_dict(), ensure_ascii=False)
    history_weeks = hist_data[-6:] if 'hist_data' in globals() else []
    chart_weeks_labels = json.dumps([f"W{w['week_num']}" for w in history_weeks])
    chart_weeks_values = json.dumps([w['total_tickets'] for w in history_weeks])

    # Рендеринг HTML
    html_output = template.render(
        planshet_tickets=planshet_tickets,
        planshet_total=planshet_total,
        week_header=f"Неделя {max_week} [{min_cls_str}]",
        total_tickets=total_tickets,
        med_hour=medHour,
        mid_hour=midHour,
        w1_t=w1_t,
        w1_m=w1_m,
        dates_reg=f"{min_reg_str} — {max_reg_str}",
        dates_cls=f"{min_cls_str} — {max_cls_str}",
        top_initiators=top_initiators,
        emo_total=emo_total,
        emo_avg=emo_avg,
        emo_max=emo_max,
        emo_tickets=emo_tickets,
        top_topics=top_topics,
        crit_tickets=crit_tickets,
        alt_total=alt_total,
        alt_tickets=alt_tickets,
        top_analysts=top_analysts,
        bad_tickets=bad_tickets,
        exc_tickets=exc_tickets,
        ai_conclusions=ai_conclusions,
        tip_text="Если устранить 3 самые частые проблемы, нагрузка на поддержку снизится минимум на 15%.",
        last30_tickets=last30_tickets,
        hardest_tickets=hardest_tickets,
        easiest_tickets=easiest_tickets,
        longest_tickets=longest_tickets,
        nom_hard_anl=nom_hard_ticket['anl'] if nom_hard_ticket is not None else 'Н/Д',
        nom_hard_txt=f"Тикет {nom_hard_ticket['id']} — сложность {nom_hard_ticket['complexity']}" if nom_hard_ticket is not None else 'Н/Д',
        nom_qty_anl=nom_qty['anl'] if nom_qty is not None else 'Н/Д',
        nom_qty_txt=f"{nom_qty['total']} решенных обращений" if nom_qty is not None else 'Н/Д',
        nom_qual_anl=nom_qual['anl'] if nom_qual is not None else 'Н/Д',
        nom_qual_txt=f"Итоговый балл {nom_qual['score']:.1f}" if nom_qual is not None else 'Н/Д',
        pie_json=pie_json,
        chart_weeks_labels=chart_weeks_labels,
        chart_weeks_values=chart_weeks_values,
        badge_class=badge_class,
        batch_tokens=total_batch_tokens['total'],
        summary_tokens=summary_tokens['total'],
        batch_model=BATCH_MODEL,
        summary_model=SUMMARY_MODEL
    )

    # Базовое имя файла
    base_filename = f"Отчет по качеству W{max_week} ({min_cls_str})"
    extension = ".html"

    drive_output_path = os.path.join(TARGET_DIR, f"{base_filename}{extension}")

    # Проверка на существование файла и генерация уникального имени (_v1, _v2, ...)
    counter = 1
    while os.path.exists(drive_output_path):
        drive_output_path = os.path.join(TARGET_DIR, f"{base_filename}_v{counter}{extension}")
        counter += 1

    # Запись готового отчета
    with open(drive_output_path, 'w', encoding='utf-8') as f:
        f.write(html_output)

    print(f"✅ Отчет успешно сохранен без перезаписи: {drive_output_path}")
    files.download(drive_output_path)
else:
    print("⏭️ ПРОПУСК: Отчет (ENABLE_MAIN = False).")

✅ Шаблон найден и загружен из: /content/drive/MyDrive/REPORTS/SPPR/template.html
✅ Отчет успешно сохранен без перезаписи: /content/drive/MyDrive/REPORTS/SPPR/result/Отчет по качеству W34 (17.08.2026).html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#ПРИЛОЖЕНИЕ 1

In [ ]:
# =====================================================================
# [ЯЧЕЙКА 2] ВЕКТОРНЫЙ ПОИСК ПОХОЖИХ ОБРАЩЕНИЙ (TF-IDF)
# =====================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 🎛️ ГЛАВНЫЙ ПЕРЕКЛЮЧАТЕЛЬ 🎛️
ENABLE_AI_ANALYSIS = True #@param {type:"boolean"}

if ENABLE_AI_ANALYSIS:
    print("🔍 Запуск алгоритма поиска похожих обращений...")

    # Заполняем пустоты
    df['desc_clean'] = df['desc'].fillna('')

    # Создаем TF-IDF матрицу
    vectorizer = TfidfVectorizer(max_features=10000, stop_words=None)
    tfidf_matrix = vectorizer.fit_transform(df['desc_clean'])

    # Вычисляем косинусное сходство
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    similar_ids_list = []
    similar_context_list = []
    SIMILARITY_THRESHOLD = 0.3 # Порог похожести

    for idx in range(len(df)):
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:4] # Топ-3

        s_ids = []
        s_context = ""

        for i, score in sim_scores:
            if score >= SIMILARITY_THRESHOLD:
                match_row = df.iloc[i]
                s_ids.append(str(match_row['id']))
                s_context += f"- [ID {match_row['id']}] Проблема: {match_row['desc'][:100]}... | Решение: {match_row['sol']}\n"

        similar_ids_list.append(", ".join(s_ids) if s_ids else "Нет аналогов")
        similar_context_list.append(s_context if s_context else "Похожих инцидентов в базе не найдено.")

    # Записываем результаты в датафрейм
    df['similar_ids'] = similar_ids_list
    df['similar_past_context'] = similar_context_list

    print("✅ Поиск завершен. Контекст прошлых решений добавлен к датасету.")
else:
    print("⏭️ ПРОПУСК: Поиск похожих обращений отключен (ENABLE_AI_ANALYSIS = False).")
    # Добавляем пустые колонки, чтобы код дальше не падал, если мы передумаем
    df['similar_ids'] = "Анализ отключен"
    df['similar_past_context'] = "Анализ отключен"

🔍 Запуск алгоритма поиска похожих обращений...
✅ Поиск завершен. Контекст прошлых решений добавлен к датасету.


In [ ]:
# =====================================================================
# [ЯЧЕЙКА 3] БЛОК АСИНХРОННОЙ ОБРАБОТКИ ИИ (С ИСТОРИЧЕСКИМ КОНТЕКСТОМ)
# =====================================================================
class TicketResult(BaseModel):
    id: str = Field(description="Идентификатор тикета")
    topic_cluster: str = Field(description="Конкретная суть сбоя/операции. Без абстракций.")
    tone_deviation_score: int = Field(description="Шкала тона от 0 до 3")
    is_emotional: bool = Field(description="True если tone_deviation_score > 0")
    emotion_argument: str = Field(description="Обоснование оценки тона")
    complexity: int = Field(description="Сложность задачи от 1 до 5")
    initial_grade: str = Field(description="Исходная оценка")
    final_grade: str = Field(description="Итоговый грейд: ПЛОХО, ХОРОШО, ОТЛИЧНО")
    grade_argument: str = Field(description="Детальная аргументация грейда и причин снижения")
    is_alternative: bool = Field(description="True если ИИ не согласен с закрытием тикета")
    is_critical_incident: bool = Field(description="True если инцидент блокирует бизнес-процесс")
    planshet: bool = Field(description="True Если Обращение относится к работе планшетов")

    # ПОЛЯ ДЛЯ АНАЛИЗА РЕШЕНИЙ
    similar_ticket_ids: str = Field(description="ID похожих обращений из переданного контекста")
    is_standard_solution: bool = Field(description="True, если предложенное решение совпадает с решениями прошлых инцидентов")
    solution_comparison: str = Field(description="Сравнительный анализ: насколько текущее решение похоже на предыдущие.")

class BatchResponse(BaseModel):
    results: list[TicketResult]

SYSTEM_INSTRUCTION = """
Вы — ведущий системный аналитик и архитектор 1С. Ваша задача — провести жесткий аудит обращений.

НОВЫЕ ЗАДАЧИ:
1. Изучите блок "Похожие прошлые инциденты". Укажите их ID в similar_ticket_ids.
2. Сравните "Текущее Решение" с решениями из прошлых инцидентов.
3. Напишите емкий сравнительный анализ (solution_comparison).
4. Установите is_standard_solution = true, если подход совпадает с историческим. Если он применил костыль или контекст аналогов пуст — false.

1. ПРАВИЛА КЛАСТЕРИЗАЦИИ (topic_cluster):
- Категорически запрещены абстрактные формулировки: "Ошибка системы", "Проблема с 1С".
- Указывайте КОНКРЕТНУЮ суть сбоя.

2. КРИТЕРИИ ОЦЕНКИ ГРЕЙДА (final_grade):
- ПЛОХО: Ручные правки БД без устранения причины, формальные отписки.
- ХОРОШО: Стандартное решение, проблема решена.
- ОТЛИЧНО: Превентивное решение, инструкция, баг-репорт.

3. ШКАЛА ЭМОЦИОНАЛЬНОГО ТОНА (tone_deviation_score): от 0 до 3.
4. СЛОЖНОСТЬ (complexity): от 1 до 5.
"""

# Функции асинхронного вызова ИИ (оставляем без изменений)
async def call_ai_provider_batch(prompt: str):
    if AI_PROVIDER == "google":
        response = await client_gemini.aio.models.generate_content(
            model=BATCH_MODEL, contents=prompt,
            config={'system_instruction': SYSTEM_INSTRUCTION, 'response_mime_type': 'application/json', 'response_schema': BatchResponse, 'temperature': 0.1}
        )
        data = json.loads(response.text).get('results', [])
        tokens = {'total': response.usage_metadata.total_token_count if response.usage_metadata else 0}
        return data, tokens
    else:
        schema_format = BatchResponse.model_json_schema()
        full_system = f"{SYSTEM_INSTRUCTION}\nОтвет строго в JSON:\n{json.dumps(schema_format, ensure_ascii=False)}"
        response = await client_deepseek_async.chat.completions.create(
            model=BATCH_MODEL,
            messages=[{"role": "system", "content": full_system}, {"role": "user", "content": prompt}],
            response_format={"type": "json_object"}, temperature=0.1
        )
        res_raw = json.loads(response.choices[0].message.content)
        data = res_raw.get('results', []) if isinstance(res_raw, dict) else res_raw
        tokens = {'total': response.usage.total_tokens if response.usage else 0}
        return data, tokens

async def process_chunk(chunk_id, chunk, sem):
    prompt_rows = []
    for _, r in chunk.iterrows():
        prompt_rows.append(
            f"ID: {r['id']} | Инициатор: {r.get('cli','')} | Аналитик: {r.get('anl','')}\n"
            f"Тема: {r.get('sec','')}\n"
            f"Описание: {r.get('desc','')}\n"
            f"Текущее Решение: {r.get('sol','')}\n"
            f"--- ПОХОЖИЕ ПРОШЛЫЕ ИНЦИДЕНТЫ ---\n"
            f"{r.get('similar_past_context', 'Нет данных')}\n"
            f"========================================"
        )
    prompt = "Оцени пакет обращений, сделай сравнительный анализ с прошлыми решениями:\n\n" + "\n".join(prompt_rows)

    async with sem:
        try:
            await asyncio.sleep(SLEEP_TIME)
            data, tokens = await call_ai_provider_batch(prompt)
            print(f"✅ Пакет {chunk_id} завершен (Токенов: {tokens['total']})")
            return data, tokens
        except Exception as e:
            print(f"⚠️ Сбой пакета {chunk_id}: {e}")
            return [], {'total': 0}

async def run_batching():
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    tasks = []
    chunks = [df.iloc[i:i+BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
    print(f"🚀 Старт анализа (Пакетов: {len(chunks)})...")

    for idx, chunk in enumerate(chunks, 1):
        tasks.append(process_chunk(idx, chunk, sem))

    results = await asyncio.gather(*tasks)
    flat_results = []
    total_toks = 0
    for res_list, toks in results:
        flat_results.extend(res_list)
        total_toks += toks['total']

    print(f"✅ Анализ ИИ завершен. Итого токенов: {total_toks}")
    return flat_results, total_toks

# === ЗАПУСК ИИ ПО УСЛОВИЮ ===
if ENABLE_AI_ANALYSIS:
    ai_results, total_batch_tokens = await run_batching()
    df_final = pd.merge(df, pd.DataFrame(ai_results), on='id', how='left')
else:
    print("⏭️ ПРОПУСК: ИИ-анализ отключен (ENABLE_AI_ANALYSIS = False).")
    df_final = df.copy() # Если ИИ отключен, финализируем исходный датафрейм

🚀 Старт анализа (Пакетов: 4)...
✅ Пакет 4 завершен (Токенов: 16517)
⚠️ Сбой пакета 2: Expecting property name enclosed in double quotes: line 626 column 6 (char 26163)
⚠️ Сбой пакета 3: Expecting ',' delimiter: line 665 column 1 (char 26163)
⚠️ Сбой пакета 1: Unterminated string starting at: line 654 column 27 (char 26604)
✅ Анализ ИИ завершен. Итого токенов: 16517


In [ ]:
# =====================================================================
# [ЯЧЕЙКА 3] БЛОК АСИНХРОННОЙ ОБРАБОТКИ ИИ (С ИСТОРИЧЕСКИМ КОНТЕКСТОМ)
# =====================================================================
class TicketResult(BaseModel):
    id: str = Field(description="Идентификатор тикета")
    topic_cluster: str = Field(description="Конкретная суть сбоя/операции. Без абстракций.")
    tone_deviation_score: int = Field(description="Шкала тона от 0 до 3")
    is_emotional: bool = Field(description="True если tone_deviation_score > 0")
    emotion_argument: str = Field(description="Обоснование оценки тона")
    complexity: int = Field(description="Сложность задачи от 1 до 5")
    initial_grade: str = Field(description="Исходная оценка")
    final_grade: str = Field(description="Итоговый грейд: ПЛОХО, ХОРОШО, ОТЛИЧНО")
    grade_argument: str = Field(description="Детальная аргументация грейда и причин снижения")
    is_alternative: bool = Field(description="True если ИИ не согласен с закрытием тикета")
    is_critical_incident: bool = Field(description="True если инцидент блокирует бизнес-процесс")
    planshet: bool = Field(description="True Если Обращение относится к работе планшетов")

    # ПОЛЯ ДЛЯ АНАЛИЗА РЕШЕНИЙ
    similar_ticket_ids: str = Field(description="ID похожих обращений из переданного контекста")
    is_standard_solution: bool = Field(description="True, если предложенное решение совпадает с решениями прошлых инцидентов")
    solution_comparison: str = Field(description="Сравнительный анализ: насколько текущее решение похоже на предыдущие.")

class BatchResponse(BaseModel):
    results: list[TicketResult]

SYSTEM_INSTRUCTION = """
Вы — ведущий системный аналитик и архитектор 1С. Ваша задача — провести жесткий аудит обращений.

НОВЫЕ ЗАДАЧИ:
1. Изучите блок "Похожие прошлые инциденты". Укажите их ID в similar_ticket_ids.
2. Сравните "Текущее Решение" с решениями из прошлых инцидентов.
3. Напишите емкий сравнительный анализ (solution_comparison).
4. Установите is_standard_solution = true, если подход совпадает с историческим. Если он применил костыль или контекст аналогов пуст — false.

1. ПРАВИЛА КЛАСТЕРИЗАЦИИ (topic_cluster):
- Категорически запрещены абстрактные формулировки: "Ошибка системы", "Проблема с 1С".
- Указывайте КОНКРЕТНУЮ суть сбоя.

2. КРИТЕРИИ ОЦЕНКИ ГРЕЙДА (final_grade):
- ПЛОХО: Ручные правки БД без устранения причины, формальные отписки.
- ХОРОШО: Стандартное решение, проблема решена.
- ОТЛИЧНО: Превентивное решение, инструкция, баг-репорт.

3. ШКАЛА ЭМОЦИОНАЛЬНОГО ТОНА (tone_deviation_score): от 0 до 3.
4. СЛОЖНОСТЬ (complexity): от 1 до 5.
"""

# Функции асинхронного вызова ИИ (оставляем без изменений)
async def call_ai_provider_batch(prompt: str):
    if AI_PROVIDER == "google":
        response = await client_gemini.aio.models.generate_content(
            model=BATCH_MODEL, contents=prompt,
            config={'system_instruction': SYSTEM_INSTRUCTION, 'response_mime_type': 'application/json', 'response_schema': BatchResponse, 'temperature': 0.1}
        )
        data = json.loads(response.text).get('results', [])
        tokens = {'total': response.usage_metadata.total_token_count if response.usage_metadata else 0}
        return data, tokens
    else:
        schema_format = BatchResponse.model_json_schema()
        full_system = f"{SYSTEM_INSTRUCTION}\nОтвет строго в JSON:\n{json.dumps(schema_format, ensure_ascii=False)}"
        response = await client_deepseek_async.chat.completions.create(
            model=BATCH_MODEL,
            messages=[{"role": "system", "content": full_system}, {"role": "user", "content": prompt}],
            response_format={"type": "json_object"}, temperature=0.1
        )
        res_raw = json.loads(response.choices[0].message.content)
        data = res_raw.get('results', []) if isinstance(res_raw, dict) else res_raw
        tokens = {'total': response.usage.total_tokens if response.usage else 0}
        return data, tokens

async def process_chunk(chunk_id, chunk, sem):
    prompt_rows = []
    for _, r in chunk.iterrows():
        prompt_rows.append(
            f"ID: {r['id']} | Инициатор: {r.get('cli','')} | Аналитик: {r.get('anl','')}\n"
            f"Тема: {r.get('sec','')}\n"
            f"Описание: {r.get('desc','')}\n"
            f"Текущее Решение: {r.get('sol','')}\n"
            f"--- ПОХОЖИЕ ПРОШЛЫЕ ИНЦИДЕНТЫ ---\n"
            f"{r.get('similar_past_context', 'Нет данных')}\n"
            f"========================================"
        )
    prompt = "Оцени пакет обращений, сделай сравнительный анализ с прошлыми решениями:\n\n" + "\n".join(prompt_rows)

    async with sem:
        try:
            await asyncio.sleep(SLEEP_TIME)
            data, tokens = await call_ai_provider_batch(prompt)
            print(f"✅ Пакет {chunk_id} завершен (Токенов: {tokens['total']})")
            return data, tokens
        except Exception as e:
            print(f"⚠️ Сбой пакета {chunk_id}: {e}")
            return [], {'total': 0}

async def run_batching():
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    tasks = []
    chunks = [df.iloc[i:i+BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
    print(f"🚀 Старт анализа (Пакетов: {len(chunks)})...")

    for idx, chunk in enumerate(chunks, 1):
        tasks.append(process_chunk(idx, chunk, sem))

    results = await asyncio.gather(*tasks)
    flat_results = []
    total_toks = 0
    for res_list, toks in results:
        flat_results.extend(res_list)
        total_toks += toks['total']

    print(f"✅ Анализ ИИ завершен. Итого токенов: {total_toks}")
    return flat_results, total_toks

# === ЗАПУСК ИИ ПО УСЛОВИЮ ===
if ENABLE_AI_ANALYSIS:
    ai_results, total_batch_tokens = await run_batching()
    df_final = pd.merge(df, pd.DataFrame(ai_results), on='id', how='left')
else:
    print("⏭️ ПРОПУСК: ИИ-анализ отключен (ENABLE_AI_ANALYSIS = False).")
    df_final = df.copy() # Если ИИ отключен, финализируем исходный датафрейм

🚀 Старт анализа (Пакетов: 4)...
✅ Пакет 4 завершен (Токенов: 16580)
✅ Пакет 1 завершен (Токенов: 30397)
⚠️ Сбой пакета 3: Expecting property name enclosed in double quotes: line 651 column 6 (char 26378)
⚠️ Сбой пакета 2: Unterminated string starting at: line 667 column 7 (char 26001)
✅ Анализ ИИ завершен. Итого токенов: 46977
